# Map of Peat Cover and Burned Area Intersection in Alaska

This notebook creates an interactive map showing where fire perimeters intersect with peat soils in Alaska.

**Data Sources:**
- Fire perimeters: `data/fire/AlaskaFireHistory_Polygons.gdb` (layer: `AK_fire_location_polygons_AKAlbersNAD83`)
- Peat cover: `data/peat/AlbersPeatMap.tif` (values 1 and 2 indicate wet/moist peat)

## 1. Import Required Libraries

In [2]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.plot import show as rioshow
from rasterstats import zonal_stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import contextily as ctx

# For better plotting
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

## 2. Define File Paths

In [6]:
from pathlib import Path

# File paths
FIRE_GDB = Path("data/fire/AlaskaFireHistory_Polygons.gdb")
FIRE_LAYER = "AK_fire_location_polygons_AKAlbersNAD83"
PEAT_RASTER = Path("data/peat/AlbersPeatMap.tif")

# Check if files exist
print(f"Fire GDB exists: {FIRE_GDB.exists()}")
print(f"Peat raster exists: {PEAT_RASTER.exists()}")

# Convert to string when needed for geopandas/rasterio
FIRE_GDB = str(FIRE_GDB)
PEAT_RASTER = str(PEAT_RASTER)

Fire GDB exists: True
Peat raster exists: True


## 3. Load Fire Perimeter Data

In [7]:
# Load fire perimeters
print(f"Loading fire layer '{FIRE_LAYER}' from geodatabase...")
fire_gdf = gpd.read_file(FIRE_GDB, layer=FIRE_LAYER)

print(f"Loaded {len(fire_gdf)} fire polygons")
print(f"CRS: {fire_gdf.crs}")
print(f"Columns: {fire_gdf.columns.tolist()}")
print(f"\nDate range: {fire_gdf.FIREYEAR.min()} to {fire_gdf.FIREYEAR.max()}")

Loading fire layer 'AK_fire_location_polygons_AKAlbersNAD83' from geodatabase...
Loaded 5714 fire polygons
CRS: EPSG:3338
Columns: ['NAME', 'RECORDNUMBER', 'ACRES', 'AFSNUMBER', 'DOFNUMBER', 'USFSNUMBER', 'ADDNUMBER', 'PERIMETERDATE', 'LATESTPERIMETER', 'SOURCE', 'SOURCEMETHOD', 'SOURCECLASS', 'AGENCYACRES', 'ACREAGEMETHOD', 'COMMENTS', 'FIREID', 'FIREYEAR', 'UPDATETIME', 'USEDONFINALREPORT', 'FPOUTDATE', 'FPMERGEDDATE', 'IRWINID', 'PRESCRIBED', 'FIRESEASONS', 'Shape_Length', 'Shape_Area', 'geometry']

Date range: 1940 to 2024


c:\Users\judith.ament\Anaconda3\envs\alaska-peat\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


In [9]:
# Save fire layer as shapefile
output_shp_path = "data/fire/fire_polygons.shp"
os.makedirs(os.path.dirname(output_shp_path), exist_ok=True)

fire_gdf.to_file(output_shp_path)
print(f"Fire layer saved to: {output_shp_path}")

C:\Users\judith.ament\AppData\Local\Temp\ipykernel_51476\3500465853.py:5: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  fire_gdf.to_file(output_shp_path)
c:\Users\judith.ament\Anaconda3\envs\alaska-peat\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'RECORDNUMBER' to 'RECORDNUMB'
  ogr_write(
c:\Users\judith.ament\Anaconda3\envs\alaska-peat\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'PERIMETERDATE' to 'PERIMETERD'
  ogr_write(
c:\Users\judith.ament\Anaconda3\envs\alaska-peat\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field PERIMETERD create as date field, though DateTime requested.
  ogr_write(
c:\Users\judith.ament\Anaconda3\envs\alaska-peat\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LATESTPERIMETER' to 'LATESTPERI'
  ogr_write(
c:\Users\judith.ament\Anaconda3\envs\alaska-peat\Lib\site-packag

Fire layer saved to: data/fire/fire_polygons.shp


c:\Users\judith.ament\Anaconda3\envs\alaska-peat\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 126503519.25830641 of field Shape_Area of feature 3540 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\judith.ament\Anaconda3\envs\alaska-peat\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 212135608.85171935 of field Shape_Area of feature 3553 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\judith.ament\Anaconda3\envs\alaska-peat\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 220279265.63725042 of field Shape_Area of feature 3580 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\judith.ament\Anaconda3\envs\alaska-peat\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 114833054.56842501 of field Shape_Area of feature 3586 not successfully written. Possibly due to

## 4. Load Peat Raster Data

In [8]:
# Load peat raster metadata
with rasterio.open(PEAT_RASTER) as src:
    peat_crs = src.crs
    peat_bounds = src.bounds
    peat_transform = src.transform
    peat_shape = src.shape
    
    # Read the peat data (may be large, so read a downsampled version for visualization)
    # Read at full resolution for now
    peat_data = src.read(1)
    
print(f"Peat raster CRS: {peat_crs}")
print(f"Peat raster shape: {peat_shape}")
print(f"Peat raster bounds: {peat_bounds}")
print(f"Unique values in peat raster: {np.unique(peat_data[peat_data > 0])}")
print(f"Peat values 1 or 2 represent wet/moist peat areas")

MemoryError: Unable to allocate 60.2 GiB for an array with shape (1, 115473, 139997) and data type float32

## 5. Ensure CRS Alignment

In [ ]:
# Check CRS alignment
if fire_gdf.crs != peat_crs:
    print(f"Reprojecting fire polygons from {fire_gdf.crs} to {peat_crs}...")
    fire_gdf = fire_gdf.to_crs(peat_crs)
else:
    print("Fire polygons and peat raster are already in the same CRS")
    
print(f"Final CRS: {fire_gdf.crs}")

## 6. Identify Fire-Peat Intersections

Use rasterstats to compute the overlap between fire polygons and peat areas.

In [ ]:
# Get pixel area for area calculations
with rasterio.open(PEAT_RASTER) as src:
    transform = src.transform
    pixel_area_m2 = abs(transform.a * -transform.e)

# Compute peat overlap for each fire polygon
print("Computing fire-peat intersections using zonal statistics...")
stats = zonal_stats(
    fire_gdf.geometry,
    PEAT_RASTER,
    stats=None,
    categorical=True,
    nodata=0,
    all_touched=False,
)

# Count peat pixels (values 1 or 2) in each fire polygon
peat_values = [1, 2]
peat_pixel_counts = []

for s in stats:
    if not s:
        peat_pixel_counts.append(0)
        continue
    cnt = sum([s.get(v, 0) for v in peat_values])
    peat_pixel_counts.append(cnt)

# Add peat information to fire GeoDataFrame
fire_gdf['peat_pixel_count'] = peat_pixel_counts
fire_gdf['peat_area_ha'] = fire_gdf['peat_pixel_count'] * (pixel_area_m2 / 10000.0)
fire_gdf['total_area_ha'] = fire_gdf.geometry.area / 10000.0
fire_gdf['has_peat'] = fire_gdf['peat_pixel_count'] > 0

# Summary statistics
n_total = len(fire_gdf)
n_with_peat = fire_gdf['has_peat'].sum()
total_peat_burned = fire_gdf['peat_area_ha'].sum()

print(f"\n--- Summary ---")
print(f"Total fire polygons: {n_total:,}")
print(f"Fire polygons intersecting peat: {n_with_peat:,} ({100*n_with_peat/n_total:.1f}%)")
print(f"Total burned peat area: {total_peat_burned:,.0f} ha")

## 7. Create a Subset for Visualization

For better visualization, let's focus on fires from recent years that intersect with peat.

In [ ]:
# Filter for fires with peat overlap
fires_on_peat = fire_gdf[fire_gdf['has_peat']].copy()

# Optional: focus on recent years for clearer visualization
RECENT_YEARS = 2000  # Change this to focus on specific time period
fires_on_peat_recent = fires_on_peat[fires_on_peat['FIREYEAR'] >= RECENT_YEARS].copy()

print(f"Fires on peat since {RECENT_YEARS}: {len(fires_on_peat_recent):,}")
print(f"Total peat burned since {RECENT_YEARS}: {fires_on_peat_recent['peat_area_ha'].sum():,.0f} ha")

## 8. Create Map: Overview of All Fires and Peat Distribution

In [ ]:
# Create a map showing peat distribution and fire perimeters
fig, ax = plt.subplots(figsize=(16, 12))

# Plot peat raster (only values 1 and 2)
with rasterio.open(PEAT_RASTER) as src:
    # Create a masked version showing only peat pixels
    peat_masked = np.where((peat_data == 1) | (peat_data == 2), 1, np.nan)
    
    rioshow(peat_masked, 
            transform=src.transform,
            ax=ax,
            cmap='YlOrBr',
            alpha=0.6,
            vmin=0,
            vmax=1,
            zorder=1)

# Plot all fire perimeters (light gray)
fire_gdf.plot(ax=ax, 
              facecolor='none', 
              edgecolor='gray', 
              linewidth=0.3, 
              alpha=0.4,
              zorder=2,
              label='All fires')

# Plot fires that intersect peat (red outline)
fires_on_peat_recent.plot(ax=ax,
                          facecolor='none',
                          edgecolor='red',
                          linewidth=0.8,
                          alpha=0.7,
                          zorder=3,
                          label=f'Fires on peat (since {RECENT_YEARS})')

# Add legend
peat_patch = mpatches.Patch(color='#d95f02', alpha=0.6, label='Peat soils')
all_fires_patch = mpatches.Patch(facecolor='none', edgecolor='gray', label='All fire perimeters')
peat_fires_patch = mpatches.Patch(facecolor='none', edgecolor='red', 
                                   label=f'Fires on peat (since {RECENT_YEARS})')
ax.legend(handles=[peat_patch, all_fires_patch, peat_fires_patch], 
          loc='upper right', fontsize=10)

ax.set_title('Alaska Peat Soils and Fire Perimeters', fontsize=16, fontweight='bold')
ax.set_xlabel('Easting (m)', fontsize=12)
ax.set_ylabel('Northing (m)', fontsize=12)

plt.tight_layout()
plt.show()

## 9. Create Zoomed Map: Detailed View of Largest Fire-Peat Intersections

In [ ]:
# Find the largest fires on peat for a detailed view
top_fires = fires_on_peat_recent.nlargest(10, 'peat_area_ha')

if len(top_fires) > 0:
    # Get bounding box of top fires
    minx, miny, maxx, maxy = top_fires.total_bounds
    
    # Add some buffer (10%)
    buffer_x = (maxx - minx) * 0.1
    buffer_y = (maxy - miny) * 0.1
    
    fig, ax = plt.subplots(figsize=(16, 12))
    
    # Plot peat raster within bounds
    with rasterio.open(PEAT_RASTER) as src:
        peat_masked = np.where((peat_data == 1) | (peat_data == 2), 1, np.nan)
        
        rioshow(peat_masked, 
                transform=src.transform,
                ax=ax,
                cmap='YlOrBr',
                alpha=0.7,
                vmin=0,
                vmax=1,
                zorder=1)
    
    # Plot the top fires with peat impact
    top_fires.plot(ax=ax,
                   column='peat_area_ha',
                   cmap='Reds',
                   edgecolor='darkred',
                   linewidth=1.5,
                   alpha=0.7,
                   legend=True,
                   legend_kwds={'label': 'Peat burned area (ha)',
                                'orientation': 'horizontal',
                                'pad': 0.05},
                   zorder=3)
    
    # Set bounds
    ax.set_xlim(minx - buffer_x, maxx + buffer_x)
    ax.set_ylim(miny - buffer_y, maxy + buffer_y)
    
    ax.set_title(f'Top 10 Fires by Peat Burned Area (since {RECENT_YEARS})', 
                 fontsize=16, fontweight='bold')
    ax.set_xlabel('Easting (m)', fontsize=12)
    ax.set_ylabel('Northing (m)', fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    # Print details about top fires
    print("\nTop 10 fires by peat burned area:")
    display(top_fires[['FIREYEAR', 'peat_area_ha', 'total_area_ha']].sort_values('peat_area_ha', ascending=False))
else:
    print(f"No fires on peat found since {RECENT_YEARS}")

## 10. Interactive Web Map with Folium

Create an interactive HTML map that can be opened in a browser.

In [ ]:
try:
    import folium
    from folium import plugins
    
    # Reproject to WGS84 for folium
    fires_wgs84 = fires_on_peat_recent.to_crs('EPSG:4326')
    
    # Get center point for map
    center_lat = fires_wgs84.geometry.centroid.y.mean()
    center_lon = fires_wgs84.geometry.centroid.x.mean()
    
    # Create base map
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=5,
        tiles='OpenStreetMap',
        control_scale=True
    )
    
    # Add fires on peat as a GeoJson layer
    folium.GeoJson(
        fires_wgs84,
        name=f'Fires on Peat (since {RECENT_YEARS})',
        style_function=lambda x: {
            'fillColor': 'red',
            'color': 'darkred',
            'weight': 2,
            'fillOpacity': 0.3,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=['FIREYEAR', 'peat_area_ha', 'total_area_ha'],
            aliases=['Fire Year:', 'Peat Burned (ha):', 'Total Area (ha):'],
            localize=True,
            sticky=False
        )
    ).add_to(m)
    
    # Add layer control
    folium.LayerControl().add_to(m)
    
    # Add title
    title_html = '''
                 <div style="position: fixed; 
                             top: 10px; left: 50px; width: 400px; height: 90px; 
                             background-color: white; border:2px solid grey; z-index:9999; 
                             font-size:14px; padding: 10px">
                 <h4>Alaska: Fires Intersecting Peat Soils</h4>
                 <p>Red polygons show fire perimeters that overlap with peat areas (since {}).</p>
                 </div>
                 '''.format(RECENT_YEARS)
    m.get_root().html.add_child(folium.Element(title_html))
    
    # Save map
    output_path = "../../outputs/plots/fire_peat_intersection_map.html"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    m.save(output_path)
    
    print(f"Interactive map saved to: {output_path}")
    print("Open this file in a web browser to explore the map interactively.")
    
    # Display in notebook
    m
    
except ImportError:
    print("Folium is not installed. Install it with: pip install folium")
    print("Skipping interactive map generation.")

## 11. Export Results

Save the fires-on-peat dataset and static maps for future reference.

In [ ]:
# Create output directory
output_dir = "../../outputs/exploratory/fire_peat_maps"
os.makedirs(output_dir, exist_ok=True)

# Save the fires-on-peat GeoDataFrame as a shapefile or GeoJSON
fires_on_peat_recent.to_file(
    os.path.join(output_dir, "fires_on_peat_recent.geojson"),
    driver='GeoJSON'
)
print(f"Saved fires-on-peat data to: {output_dir}/fires_on_peat_recent.geojson")

# Save summary statistics to CSV
summary_df = fires_on_peat_recent[['FIREYEAR', 'peat_area_ha', 'total_area_ha']].copy()
summary_df = summary_df.sort_values('peat_area_ha', ascending=False)
summary_df.to_csv(os.path.join(output_dir, "fires_on_peat_summary.csv"), index=False)
print(f"Saved summary statistics to: {output_dir}/fires_on_peat_summary.csv")

print("\n✓ All outputs saved successfully!")